# IndoBERT NER Fine-tuning Pipeline
Pipeline ini disiapkan untuk dieksekusi di Google Colab. 
Jalankan cell secara berurutan untuk menyiapkan environment, me-write file pipeline, dan menjalankan training / inference.

In [ ]:
# 1. Setup Direktori
!mkdir -p src scripts data/raw data/processed models/checkpoints output

### Menulis file `requirement.txt`

In [ ]:
%%writefile requirement.txt
spacy==3.7.4
transformers==4.40.2
# PyTorch CUDA 12.1 — install via:
# pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
torch==2.2.2
datasets==2.18.0
seqeval==1.2.2
sentencepiece==0.2.0
accelerate==0.29.3

pandas>=2.0.0
pyarrow>=14.0.0

pymupdf==1.23.26
Pillow==10.3.0
numpy==1.26.4
opencv-python==4.9.0.80

paddlepaddle==2.6.2
paddleocr==2.7.3
protobuf==3.20.0

# ── LayoutLMv3 untuk table detection ────────────────────────
# Model: microsoft/layoutlmv3-base (diunduh otomatis dari HuggingFace)
# Tidak ada package tambahan — sudah tercover oleh transformers>=4.25
# Pastikan juga:
#   pip install torchvision  (sudah via PyTorch install di atas)
#   pip install Pillow       (sudah di atas)
#   pip install detectron2   (opsional, hanya jika pakai image backbone)


### Menulis file `config.py`

In [ ]:
%%writefile config.py
"""
config.py
Konfigurasi terpusat untuk project NER Surat Indonesia.
"""

from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict


# ─────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────
BASE_DIR   = Path(__file__).parent
DATA_DIR   = BASE_DIR / "data"
RAW_DIR    = DATA_DIR / "raw"
PROC_DIR   = DATA_DIR / "processed"
MODEL_DIR  = BASE_DIR / "models"
CKPT_DIR   = MODEL_DIR / "checkpoints"
OUTPUT_DIR = BASE_DIR / "output"

for _d in [RAW_DIR, PROC_DIR, CKPT_DIR, OUTPUT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────
# NER Labels
# ─────────────────────────────────────────────────────────────
# Label lokal (dokumen surat Indonesia)
LOCAL_LABELS: List[str] = [
    "NOMOR_SURAT",   # Label 1
    "JENIS_DOKUMEN", # Label 2
    "TANGGAL",       # Label 3
    "PENGIRIM",      # Label 4
    "PENERIMA",      # Label 5
    "PERIHAL",       # Label 6
    "ISI",           # Label 7  — isi utama dokumen (rule-based + NER)
    "TABEL",         # Label 8  — blok tabel (rule-based hybrid + LayoutLMv3)
    "LOKASI",        # Label 9  (dataset baru)
]

# Label dari HuggingFace indo-ner-dataset (general NER)
HF_LABELS: List[str] = [
    "PER",           # Person
    "LOC",           # Location
    "ORG",           # Organization
    "TIME",          # Temporal expression
    "TIT",           # Title
]

# Gabungan semua label (local + HF)
# LABELS: List[str] = LOCAL_LABELS + HF_LABELS
LABELS: List[str] = LOCAL_LABELS

# BIO tagging scheme: B-<label>, I-<label>, O
BIO_LABELS: List[str] = ["O"] + [
    prefix + label
    for label in LABELS
    for prefix in ("B-", "I-")
]

LABEL2ID: Dict[str, int] = {lbl: i for i, lbl in enumerate(BIO_LABELS)}
ID2LABEL: Dict[int, str] = {i: lbl for lbl, i in LABEL2ID.items()}
NUM_LABELS: int = len(BIO_LABELS)


# ─────────────────────────────────────────────────────────────
# Model
# ─────────────────────────────────────────────────────────────
@dataclass
class ModelConfig:
    pretrained_model: str  = "indobenchmark/indobert-base-p1"
    max_length: int        = 512
    num_labels: int        = NUM_LABELS
    dropout: float         = 0.1
    fine_tuned_path: str   = str(CKPT_DIR / "indobert-ner-finetuned")


# ─────────────────────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────────────────────
@dataclass
class TrainConfig:
    batch_size: int        = 16      # batch 16 untuk training
    eval_batch_size: int   = 16
    epochs: int            = 10
    learning_rate: float   = 2e-5
    weight_decay: float    = 0.01
    warmup_ratio: float    = 0.1
    gradient_clip: float   = 1.0
    seed: int              = 42
    train_split: float     = 0.8
    val_split: float       = 0.1
    # test_split             = 0.1  (sisa)
    save_steps: int        = 500
    eval_steps: int        = 500
    logging_steps: int     = 100
    fp16: bool             = True    # FP16 aktif: hemat ~50% VRAM di GPU NVIDIA


# ─────────────────────────────────────────────────────────────
# OCR / PDF
# ─────────────────────────────────────────────────────────────
@dataclass
class PDFConfig:
    # Threshold jumlah karakter untuk memutuskan pdf "pure" vs "scanned"
    text_char_threshold: int   = 50
    # DPI render untuk scanned PDF sebelum PaddleOCR
    render_dpi: int            = 200
    # Bahasa PaddleOCR
    ocr_lang: str              = "id"          # Indonesian
    ocr_use_gpu: bool          = False
    # Ukuran max gambar (px) sebelum OCR
    max_img_size: int          = 4096


# ─────────────────────────────────────────────────────────────
# Singleton instances
# ─────────────────────────────────────────────────────────────
model_cfg = ModelConfig()
train_cfg = TrainConfig()
pdf_cfg   = PDFConfig()


### Menulis file `src/model.py`

In [ ]:
%%writefile src/model.py
"""
src/model.py
Model IndoBERT fine-tuned untuk NER Token Classification.
Menggunakan BertForTokenClassification dari HuggingFace.
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
from loguru import logger
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    BertConfig,
    BertForTokenClassification,
)

from config import ID2LABEL, LABEL2ID, NUM_LABELS, model_cfg


# ─────────────────────────────────────────────────────────────
# Model builder
# ─────────────────────────────────────────────────────────────
def build_model(
    pretrained: str      = model_cfg.pretrained_model,
    num_labels: int      = NUM_LABELS,
    dropout: float       = model_cfg.dropout,
    from_checkpoint: Optional[str] = None,
) -> BertForTokenClassification:
    """
    Bangun model IndoBERT untuk token classification (NER).
    
    Args:
        pretrained      : HuggingFace model ID atau path lokal.
        num_labels      : Jumlah kelas NER (termasuk O & BIO prefix).
        dropout         : Dropout pada classifier head.
        from_checkpoint : Jika ada, load weights dari checkpoint lokal.
    
    Returns:
        model : BertForTokenClassification siap train/infer.
    """
    if from_checkpoint and Path(from_checkpoint).exists():
        logger.info(f"Memuat model dari checkpoint: {from_checkpoint}")
        model = AutoModelForTokenClassification.from_pretrained(
            from_checkpoint,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        )
    else:
        logger.info(f"Inisialisasi model dari pretrained: {pretrained}")
        config = AutoConfig.from_pretrained(
            pretrained,
            num_labels=num_labels,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
        )
        model = AutoModelForTokenClassification.from_pretrained(
            pretrained,
            config=config,
            ignore_mismatched_sizes=True,
        )

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Model siap | Total params: {total_params:,} | Trainable: {trainable_params:,}")

    return model


# ─────────────────────────────────────────────────────────────
# Utility: freeze/unfreeze layers
# ─────────────────────────────────────────────────────────────
def freeze_bert_layers(model: BertForTokenClassification, n_layers: int = 6):
    """
    Freeze n layer pertama BERT (embedding + beberapa encoder layer).
    Berguna untuk fine-tuning ringan dengan dataset kecil.
    """
    # Freeze embeddings
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False

    # Freeze n_layers pertama encoder
    for i, layer in enumerate(model.bert.encoder.layer):
        if i < n_layers:
            for param in layer.parameters():
                param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Frozen {n_layers} BERT layers. Trainable params: {trainable:,}")


def unfreeze_all(model: nn.Module):
    """Unfreeze semua parameter model."""
    for param in model.parameters():
        param.requires_grad = True
    logger.info("Semua parameter di-unfreeze.")


# ─────────────────────────────────────────────────────────────
# Device helper
# ─────────────────────────────────────────────────────────────
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        logger.info(f"Menggunakan GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        logger.info("Menggunakan Apple MPS (M1/M2)")
    else:
        dev = torch.device("cpu")
        logger.info("Menggunakan CPU")
    return dev


### Menulis file `src/dataset.py`

In [ ]:
%%writefile src/dataset.py
from __future__ import annotations

import json
import random
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import spacy
from loguru import logger
from transformers import AutoTokenizer

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

from config import LABEL2ID, LABELS, model_cfg, train_cfg


# ─────────────────────────────────────────────────────────────
# spaCy tokenizer (multi-language, lightweight)
# ─────────────────────────────────────────────────────────────
_nlp: Optional[spacy.Language] = None

def _get_nlp() -> spacy.Language:
    global _nlp
    if _nlp is None:
        try:
            _nlp = spacy.load("xx_ent_wiki_sm")
        except OSError:
            logger.warning("Model spaCy 'xx_ent_wiki_sm' belum di-install. Menggunakan blank.")
            _nlp = spacy.blank("id")
    return _nlp


# ─────────────────────────────────────────────────────────────
# Tokenizer IndoBERT
# ─────────────────────────────────────────────────────────────
_tokenizer: Optional[AutoTokenizer] = None

def get_tokenizer() -> AutoTokenizer:
    global _tokenizer
    if _tokenizer is None:
        logger.info(f"Memuat tokenizer: {model_cfg.pretrained_model}")
        _tokenizer = AutoTokenizer.from_pretrained(model_cfg.pretrained_model)
    return _tokenizer


# ─────────────────────────────────────────────────────────────
# Raw JSON loading
# ─────────────────────────────────────────────────────────────
def load_json_dataset(path: str | Path) -> List[Dict[str, Any]]:
    """Load file JSON dataset."""
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    logger.info(f"Dataset dimuat: {len(data)} sampel dari {path.name}")
    return data


def load_hf_indo_ner_dataset(split: str = "train") -> List[Dict[str, Any]]:
    """
    Load HuggingFace treamyracle/indo-ner-dataset.
    
    Format dataset HF:
      - text: str — kalimat input
      - entities: List[{"start": int, "end": int, "label": str}]
        Label: PER, LOC, ORG, TIME, TIT
    
    Args:
        split: "train" (dataset ini hanya punya split train)
    
    Returns:
        List[Dict] dengan format training-ready (sudah tokenized + aligned)
    """
    try:
        from datasets import load_dataset as hf_load_dataset
    except ImportError:
        raise ImportError(
            "datasets library diperlukan untuk load HF dataset. "
            "Install dengan: pip install datasets"
        )
    
    logger.info(f"Loading HF dataset 'treamyracle/indo-ner-dataset' split='{split}'...")
    try:
        ds = hf_load_dataset("treamyracle/indo-ner-dataset", split=split)
        logger.info(f"  HF dataset loaded: {len(ds)} rows")
    except Exception as e:
        logger.error(f"Error loading HF dataset: {e}")
        raise
    
    # Convert offset-based entities ke BIO tags, lalu tokenize+align
    samples = []
    skipped = 0
    
    for idx, item in enumerate(ds):
        try:
            text = item.get("text", "")
            raw_entities = item.get("entities", [])
            
            if not text or not text.strip():
                skipped += 1
                continue
            
            # Convert HF entity format ke internal format
            # HF: {"start": int, "end": int, "label": str}
            # Internal: {"label": str, "start": int, "end": int, "value": str}
            entities = []
            for ent in raw_entities:
                start = ent["start"]
                end = ent["end"]
                label = ent["label"]
                value = text[start:end]
                entities.append({
                    "label": label,
                    "start": start,
                    "end": end,
                    "value": value,
                })
            
            # BIO tagging via spaCy tokenizer + char-level alignment
            tokens, bio_tags = create_bio_tags(text, entities)
            
            if not tokens:
                skipped += 1
                continue
            
            # Tokenize dengan IndoBERT + align labels
            encoded = tokenize_and_align_labels(
                tokens=tokens,
                bio_tags=bio_tags,
                max_length=model_cfg.max_length,
            )
            samples.append(encoded)
            
        except Exception as e:
            if idx < 5:
                logger.warning(f"Error processing HF row {idx}: {e}")
            skipped += 1
            continue
    
    logger.info(f"HF dataset '{split}': berhasil {len(samples)}, skip {skipped}")
    return samples


def load_dataset_combined(
    hf_only: bool = False,
    local_path: Optional[str | Path] = None,
    hf_split: str = "train",
) -> List[Dict[str, Any]]:
    """
    Load dataset dengan opsi:
    1. HF only (treamyracle/indo-ner-dataset)
    2. Local only (JSON file)
    3. Combined (HF + Local)
    
    Args:
        hf_only: Jika True, load hanya HF. Jika False, combine dengan local.
        local_path: Path ke local JSON dataset (pakai jika hf_only=False)
        hf_split: "train" atau "test"
    
    Returns:
        List[Dict] combined training samples
    """
    samples = []
    
    # Load HF dataset
    try:
        hf_samples = load_hf_indo_ner_dataset(split=hf_split)
        samples.extend(hf_samples)
        logger.info(f"HF samples: +{len(hf_samples)}")
    except Exception as e:
        logger.warning(f"Gagal load HF dataset: {e}")
        if hf_only:
            raise
    
    # Load local jika ada
    if not hf_only and local_path:
        try:
            local_data = load_json_dataset(local_path)
            local_samples = build_training_samples(local_data)
            samples.extend(local_samples)
            logger.info(f"Local samples: +{len(local_samples)}")
        except Exception as e:
            logger.warning(f"Gagal load local dataset: {e}")
    
    logger.info(f"Total combined: {len(samples)} sampel")
    return samples


def hf_format_to_spacy(
    tokens: List[str],
    bio_tags: List[str],
    labels: List[str] = None,
) -> Tuple[str, List[Tuple[int, int, str]]]:
    """
    Convert HF NER format (tokens + BIO tags) ke spaCy format (text + entities dengan offset).
    
    Args:
        tokens: List[str] — word tokens
        bio_tags: List[str] — BIO tags ["B-LOC", "I-LOC", "O", ...]
        labels: List[str] — list label names (subset dari BIO tags)
    
    Returns:
        (text, entities) — text adalah reconstructed dari tokens
                         — entities adalah list (start, end, label)
    """
    if labels is None:
        labels = LABELS
    
    # Reconstruct text dari tokens
    text = " ".join(tokens)
    
    # Parse BIO tags → entities
    entities = []
    current_label = None
    current_start = None
    char_pos = 0
    
    for i, (token, tag) in enumerate(zip(tokens, bio_tags)):
        # Handle tag
        if tag.startswith("B-"):
            # Mulai entitas baru
            if current_label is not None:
                # End entitas sebelumnya
                end = char_pos
                if current_label in labels:
                    entities.append((current_start, end, current_label))
            
            current_label = tag[2:]  # B-LOC → LOC
            current_start = char_pos
        
        elif tag.startswith("I-"):
            # Continue entitas
            if current_label is None:
                # Malformed: I- tanpa B-, treat as B-
                if current_label is not None and current_label in labels:
                    entities.append((current_start, char_pos, current_label))
                current_label = tag[2:]
                current_start = char_pos
        
        else:
            # tag == "O"
            if current_label is not None:
                # End entitas
                end = char_pos
                if current_label in labels:
                    entities.append((current_start, end, current_label))
                current_label = None
        
        # Move char position forward
        char_pos += len(token)
        if i < len(tokens) - 1:
            char_pos += 1  # space token separator
    
    # Handle last entity
    if current_label is not None:
        end = char_pos
        if current_label in labels:
            entities.append((current_start, end, current_label))
    
    return text, entities


def load_hf_nergrit_as_spacy_format(split: str = "train") -> List[Tuple[str, Dict]]:
    """
    Load HF treamyracle/indo-ner-dataset dan convert langsung ke spaCy format.
    
    Returns:
        List[(text, {"entities": [(start, end, label), ...]})]
    """
    try:
        from datasets import load_dataset as hf_load_dataset
    except ImportError:
        raise ImportError("datasets library diperlukan. Install: pip install datasets")
    
    logger.info(f"Loading HF dataset untuk spaCy format, split='{split}'...")
    ds = hf_load_dataset("treamyracle/indo-ner-dataset", split=split)
    
    spacy_data = []
    
    for item in ds:
        text = item.get("text", "")
        raw_entities = item.get("entities", [])
        
        if not text or not raw_entities:
            continue
        
        # Convert HF entity format ke spaCy tuple format
        entities = []
        for ent in raw_entities:
            start = ent["start"]
            end = ent["end"]
            label = ent["label"]
            entities.append((start, end, label))
        
        if entities:
            spacy_data.append((text, {"entities": entities}))
    
    logger.info(f"HF dataset '{split}' converted to spaCy format: {len(spacy_data)} samples")
    return spacy_data


# ─────────────────────────────────────────────────────────────
# Robust matching helpers (untuk OCR noisy)
# ─────────────────────────────────────────────────────────────
def normalize_loose(s: str) -> str:
    """
    Normalisasi longgar:
    - uppercase
    - hapus whitespace
    - hapus punctuation umum
    """
    if not s:
        return ""

    s = s.upper()
    s = re.sub(r'[\r\n\t]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'[.,:;()\[\]{}\-_/]', '', s)
    s = re.sub(r'\s+', '', s)
    return s


def build_normalized_mapping(text: str) -> Tuple[str, List[int]]:
    """
    Bangun teks ternormalisasi + mapping index normalized -> original text index
    agar bisa cari substring secara longgar tapi tetap dapat span original.
    """
    norm_chars = []
    mapping = []

    for idx, ch in enumerate(text):
        up = ch.upper()

        # skip punctuation ringan
        if re.match(r'[.,:;()\[\]{}\-_/]', up):
            continue

        # skip whitespace
        if up.isspace():
            continue

        norm_chars.append(up)
        mapping.append(idx)

    return "".join(norm_chars), mapping


def find_span_robust(text: str, value: str) -> Optional[Tuple[int, int]]:
    """
    Cari span value di text secara robust:
    1) exact/case-insensitive match
    2) loose match (ignore whitespace + punctuation)
    """
    if not text or not value:
        return None

    value_str = str(value)

    # 1) exact case-insensitive
    idx = text.lower().find(value_str.lower())
    if idx >= 0:
        return idx, idx + len(value_str)

    # 2) loose match
    loose_value = normalize_loose(value_str)
    loose_text, mapping = build_normalized_mapping(text)

    idx2 = loose_text.find(loose_value)
    if idx2 >= 0:
        end2 = idx2 + len(loose_value) - 1
        if idx2 < len(mapping) and end2 < len(mapping):
            start_orig = mapping[idx2]
            end_orig = mapping[end2] + 1
            return start_orig, end_orig

    return None


# ─────────────────────────────────────────────────────────────
# Format detection & normalization
# ─────────────────────────────────────────────────────────────
def normalize_sample(sample: Dict[str, Any]) -> Dict[str, Any]:
    """
    Normalisasi berbagai format JSON menjadi format internal:
    {
        "text": str,
        "entities": [{"label": str, "start": int, "end": int, "value": str}]
    }

    Mendukung:
    1. Format dengan "entities" list (offset-based)
    2. Format flat label-value dict
    3. Format nested labels dengan text kosong
       → Buat dummy text dari concatenate label values
    """
    # Format 1: sudah ada "entities" dengan offset
    if "entities" in sample and "text" in sample:
        return sample

    text = sample.get("raw_text", sample.get("text", ""))
    entities = []

    # Support 2 format:
    # - flat: sample["NOMOR_SURAT"]
    # - nested: sample["labels"]["NOMOR_SURAT"]
    labels_dict = sample.get("labels", {}) if isinstance(sample.get("labels"), dict) else {}

    def get_label_value(label: str):
        if label in sample:
            return sample.get(label, "")
        return labels_dict.get(label, "")

    # Jika text kosong, coba buat dari label values
    if not text or text.strip() == "":
        # Kumpulkan semua nilai label sebagai dummy text
        all_values = []
        for label in LABELS:
            value = get_label_value(label)
            if value:
                all_values.append(str(value))
        text = " ".join(all_values) if all_values else ""

    # Extract entities
    for label in LABELS:
        value = get_label_value(label)
        if value and text:
            value_str = str(value)

            span = find_span_robust(text, value_str)
            if span is not None:
                start, end = span
                entities.append({
                    "label": label,
                    "start": start,
                    "end": end,
                    "value": value_str,
                })

    return {"text": text, "entities": entities}


# ─────────────────────────────────────────────────────────────
# BIO tagging
# ─────────────────────────────────────────────────────────────
def create_bio_tags(text: str, entities: List[Dict]) -> Tuple[List[str], List[str]]:
    """
    Hasilkan pasangan (tokens, bio_tags) menggunakan spaCy tokenizer.

    Returns:
        tokens   : List[str] token
        bio_tags : List[str] tag BIO
    """
    nlp = _get_nlp()
    doc = nlp(text)
    tokens = [tok.text for tok in doc]

    # Buat char-level tag untuk setiap karakter
    char_tags = ["O"] * len(text)

    # Sort entities agar tidak tumpang tindih
    sorted_ents = sorted(entities, key=lambda e: e["start"])

    for ent in sorted_ents:
        start = ent["start"]
        end   = ent["end"]
        label = ent["label"]

        if label not in LABELS:
            continue

        # Clamp ke batas text
        start = max(0, min(start, len(text)))
        end = max(start, min(end, len(text)))

        # Hindari overwrite entity lama kalau overlap
        for i in range(start, end):
            if char_tags[i] != "O":
                continue
            if i == start:
                char_tags[i] = f"B-{label}"
            else:
                char_tags[i] = f"I-{label}"

    # Map char tags → token tags (ambil tag dari karakter pertama token)
    bio_tags = []
    char_offset = 0

    for tok in doc:
        tok_start = text.find(tok.text, char_offset)

        # fallback kalau token repeated / mismatch
        if tok_start == -1:
            tok_start = text.find(tok.text)

        if tok_start == -1:
            bio_tags.append("O")
            continue

        # Cari first non-space char dalam token span
        token_real_start = tok_start
        while token_real_start < len(text) and text[token_real_start].isspace():
            token_real_start += 1

        if token_real_start >= len(text):
            bio_tags.append("O")
        else:
            bio_tags.append(char_tags[token_real_start])

        char_offset = tok_start + len(tok.text)

    return tokens, bio_tags


# ─────────────────────────────────────────────────────────────
# Tokenize + align labels (untuk BERT sub-word)
# ─────────────────────────────────────────────────────────────
def tokenize_and_align_labels(
    tokens: List[str],
    bio_tags: List[str],
    max_length: int = 512,
) -> Dict[str, Any]:
    """
    Tokenisasi dengan IndoBERT dan align BIO labels ke sub-word tokens.
    Label -100 → diabaikan oleh CrossEntropyLoss.
    """
    tokenizer = get_tokenizer()

    tokenized = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )

    word_ids = tokenized.word_ids()
    label_ids = []
    prev_word = None

    for wid in word_ids:
        if wid is None:
            label_ids.append(-100)  # [CLS], [SEP], [PAD]
        elif wid != prev_word:
            # token pertama kata
            label_ids.append(LABEL2ID.get(bio_tags[wid], 0))
        else:
            # sub-word berikutnya: set I-label agar konsisten
            current_tag = bio_tags[wid]
            if current_tag.startswith("B-"):
                current_tag = "I-" + current_tag[2:]
            label_ids.append(LABEL2ID.get(current_tag, -100))

        prev_word = wid

    tokenized["labels"] = label_ids
    return dict(tokenized)


# ─────────────────────────────────────────────────────────────
# Full pipeline: JSON → training-ready dicts
# ─────────────────────────────────────────────────────────────
def build_training_samples(raw_data: List[Dict]) -> List[Dict]:
    """Konversi seluruh dataset JSON ke list dict siap training."""
    samples = []
    entity_stats = {label: 0 for label in LABELS}

    for i, item in enumerate(raw_data):
        try:
            norm = normalize_sample(item)

            if not norm["text"].strip():
                logger.warning(f"Sampel #{i} memiliki teks kosong, dilewati.")
                continue

            # hitung entity sebelum tokenisasi (debug)
            for ent in norm["entities"]:
                if ent["label"] in entity_stats:
                    entity_stats[ent["label"]] += 1

            tokens, bio_tags = create_bio_tags(norm["text"], norm["entities"])
            encoded = tokenize_and_align_labels(
                tokens, bio_tags, max_length=model_cfg.max_length
            )
            samples.append(encoded)

        except Exception as e:
            logger.error(f"Error pada sampel #{i}: {e}")

    logger.info(f"Berhasil diproses: {len(samples)}/{len(raw_data)} sampel")
    logger.info("Distribusi entity hasil normalize:")
    for label in LABELS:
        logger.info(f"  {label:<20} {entity_stats[label]}")

    return samples


# ─────────────────────────────────────────────────────────────
# Train / Val / Test split
# ─────────────────────────────────────────────────────────────
def split_dataset(
    samples: List[Dict],
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """Split dataset secara acak menjadi train/val/test."""
    random.seed(seed)
    shuffled = samples.copy()
    random.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train = shuffled[:n_train]
    val = shuffled[n_train : n_train + n_val]
    test = shuffled[n_train + n_val :]

    logger.info(f"Split dataset → Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")
    return train, val, test


# ─────────────────────────────────────────────────────────────
# PyTorch Dataset wrapper
# ─────────────────────────────────────────────────────────────
try:
    import torch
    from torch.utils.data import Dataset

    class NERDataset(Dataset):
        def __init__(self, samples: List[Dict]):
            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx) -> Dict[str, Any]:
            item = self.samples[idx]
            return {
                "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
                "token_type_ids": torch.tensor(
                    item.get("token_type_ids", [0] * len(item["input_ids"])),
                    dtype=torch.long
                ),
                "labels": torch.tensor(item["labels"], dtype=torch.long),
            }

except ImportError:
    logger.warning("PyTorch tidak terinstall. NERDataset tidak tersedia.")
    NERDataset = None


### Menulis file `src/table_detector.py`

In [ ]:
%%writefile src/table_detector.py
"""
src/table_detector.py
Deteksi Tabel menggunakan LayoutLMv3 (microsoft/layoutlmv3-base).

LayoutLMv3 memahami dokumen secara multimodal:
  - Teks (words)
  - Layout (bounding boxes dalam koordinat [0,1000])
  - Gambar (patch visual dari halaman)

Alur deteksi:
  1. Ekstrak words + bounding boxes dari PDF via PyMuPDF
  2. Feed ke LayoutLMv3Processor + LayoutLMv3ForTokenClassification
  3. Prediksi label per token → kumpulkan span label "TABEL"
  4. Kembalikan sebagai List[TableSpan] dengan confidence & teks blok tabel

Fallback:
  - Jika PDF scanned atau bbox tidak tersedia, gunakan dummy bbox
    berdasarkan posisi relatif teks (line-number based normalization)
  - Jika LayoutLM tidak terinstall, raise ImportError yang informatif

Model default : microsoft/layoutlmv3-base  (zero-shot atau fine-tuned)
                Dapat di-override via checkpoint lokal yang sudah ditraining
                dengan label B-TABEL / I-TABEL / O
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from loguru import logger


# ─────────────────────────────────────────────────────────────
# Konstanta
# ─────────────────────────────────────────────────────────────
DEFAULT_LAYOUTLM_MODEL = "microsoft/layoutlmv3-base"
LAYOUTLM_MAX_LEN       = 512          # max sequence length LayoutLMv3
BBOX_NORM              = 1000         # koordinat bbox LayoutLM: 0–1000
MIN_TABLE_TOKENS       = 4            # minimal token ber-label TABEL untuk dianggap span valid
TABLE_LABEL_KEYWORDS   = {"TABEL", "TABLE", "B-TABEL", "I-TABEL"}


# ─────────────────────────────────────────────────────────────
# Rule-based patterns (dijalankan sebelum transformer)
# ─────────────────────────────────────────────────────────────
import re as _re

# Baris yang mengandung karakter | (tabel markdown / ASCII art)
_RULE_PIPE_ROW     = _re.compile(r'\|.{2,}\|')
# Spasi multiple sebagai pemisah kolom (min 3 spasi, min 2 kemunculan per baris)
_RULE_SPACE_COL    = _re.compile(r'(?:[^\n]+ {3,}){2,}[^\n]+')
# Garis pemisah tabel
_RULE_SEPARATOR    = _re.compile(r'^[\-=+]{5,}$', _re.MULTILINE)
# Header tabel umum dokumen Indonesia
_RULE_TABLE_HEADER = _re.compile(
    r'(?:no\.?|nomor)\s*[.|)]?\s+(?:nama|uraian|keterangan|kegiatan|jenis|item|barang)',
    _re.IGNORECASE,
)
# Pola jumlah/harga dalam baris (kuat indikasi tabel)
_RULE_AMOUNT_ROW   = _re.compile(
    r'(?:Rp\.?|IDR)\s*[\d.,]+|[\d.,]+\s*(?:unit|pcs|kg|liter|buah|lembar)',
    _re.IGNORECASE,
)


# ─────────────────────────────────────────────────────────────
# Data model
# ─────────────────────────────────────────────────────────────
@dataclass
class TableSpan:
    """Representasi satu blok tabel yang terdeteksi."""
    start:      int           # indeks karakter dalam teks asli
    end:        int
    text:       str           # teks blok tabel
    page:       int           # nomor halaman (1-based)
    source:     str           # "layoutlm" | "layoutlm_fallback"
    confidence: float = 1.0  # rata-rata confidence skor prediksi

    def to_dict(self) -> Dict[str, Any]:
        return {
            "start":      self.start,
            "end":        self.end,
            "text":       self.text,
            "page":       self.page,
            "source":     self.source,
            "confidence": round(self.confidence, 4),
        }


# ─────────────────────────────────────────────────────────────
# Singleton: LayoutLMv3 model + processor
# ─────────────────────────────────────────────────────────────
_layoutlm_model     = None
_layoutlm_processor = None
_layoutlm_device    = None


def _load_layoutlm(model_path: str = DEFAULT_LAYOUTLM_MODEL) -> None:
    """
    Load LayoutLMv3 processor dan model ke memori.
    Dipanggil sekali (singleton pattern).
    """
    global _layoutlm_model, _layoutlm_processor, _layoutlm_device

    if _layoutlm_model is not None:
        return  # sudah dimuat

    try:
        import torch
        from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
    except ImportError:
        raise ImportError(
            "transformers tidak ditemukan. Pastikan sudah menginstall: "
            "pip install transformers"
        )

    try:
        from src.model import get_device
        _layoutlm_device = get_device()
    except Exception:
        import torch
        _layoutlm_device = torch.device("cpu")

    logger.info(f"[LayoutLM] Memuat model dari: {model_path}")

    _layoutlm_processor = LayoutLMv3Processor.from_pretrained(
        model_path,
        apply_ocr=False,   # kita supply words + bboxes sendiri dari PyMuPDF
    )

    _layoutlm_model = LayoutLMv3ForTokenClassification.from_pretrained(
        model_path,
    ).to(_layoutlm_device)
    _layoutlm_model.eval()

    logger.info(f"[LayoutLM] Model siap di {_layoutlm_device}")


# ─────────────────────────────────────────────────────────────
# ① Ekstraksi words + bbox dari PDF (PyMuPDF)
# ─────────────────────────────────────────────────────────────
@dataclass
class WordBox:
    word:   str
    x0:     float  # koordinat asli (pt)
    y0:     float
    x1:     float
    y1:     float
    page:   int
    char_start: int = 0   # offset karakter dalam full_text
    char_end:   int = 0


def extract_words_and_boxes(
    pdf_path: str | Path,
) -> Tuple[List[WordBox], int]:
    """
    Ekstrak setiap kata beserta bounding box-nya dari PDF menggunakan PyMuPDF.

    Returns:
        (word_boxes, page_count)
        word_boxes : List[WordBox] — semua kata dari semua halaman
        page_count : jumlah halaman
    """
    try:
        import fitz
    except ImportError:
        raise ImportError("PyMuPDF tidak terinstall. pip install pymupdf")

    pdf_path  = Path(pdf_path)
    doc       = fitz.open(str(pdf_path))
    all_words: List[WordBox] = []
    char_offset = 0

    for page_num, page in enumerate(doc, start=1):
        page_width  = page.rect.width  or 1
        page_height = page.rect.height or 1

        # get_text("words") → list of (x0,y0,x1,y1,word,block_no,line_no,word_no)
        raw_words = page.get_text("words")

        for w in raw_words:
            x0, y0, x1, y1, text = w[0], w[1], w[2], w[3], w[4]
            text = text.strip()
            if not text:
                continue

            wb = WordBox(
                word       = text,
                x0         = x0,
                y0         = y0,
                x1         = x1,
                y1         = y1,
                page       = page_num,
                char_start = char_offset,
                char_end   = char_offset + len(text),
            )
            all_words.append(wb)
            char_offset += len(text) + 1  # +1 untuk spasi

    doc.close()
    logger.debug(f"[LayoutLM] Diekstrak {len(all_words)} kata dari {doc.page_count} halaman")
    return all_words, doc.page_count


def normalize_bbox(
    x0: float, y0: float, x1: float, y1: float,
    page_width: float, page_height: float,
) -> List[int]:
    """
    Normalisasi bbox ke rentang [0, 1000] sesuai format LayoutLMv3.
    """
    def clamp(v: float, lo: float = 0.0, hi: float = 1000.0) -> int:
        return int(max(lo, min(hi, v)))

    return [
        clamp(x0 / page_width  * BBOX_NORM),
        clamp(y0 / page_height * BBOX_NORM),
        clamp(x1 / page_width  * BBOX_NORM),
        clamp(y1 / page_height * BBOX_NORM),
    ]


def build_page_inputs(
    word_boxes: List[WordBox],
    pdf_path: str | Path,
) -> List[Dict]:
    """
    Kelompokkan WordBox per halaman, buat input dict per halaman.
    Setiap dict punya: words, boxes, page_num, word_boxes_ref

    Returns:
        List[dict] — satu dict per halaman
    """
    try:
        import fitz
    except ImportError:
        raise ImportError("PyMuPDF tidak terinstall.")

    doc = fitz.open(str(pdf_path))
    page_dims: Dict[int, Tuple[float, float]] = {}
    for page_num, page in enumerate(doc, start=1):
        page_dims[page_num] = (page.rect.width or 1, page.rect.height or 1)
    doc.close()

    # Kelompokkan per halaman
    from collections import defaultdict
    per_page: Dict[int, List[WordBox]] = defaultdict(list)
    for wb in word_boxes:
        per_page[wb.page].append(wb)

    page_inputs = []
    for pnum in sorted(per_page.keys()):
        wbs     = per_page[pnum]
        pw, ph  = page_dims.get(pnum, (595, 842))  # default A4
        words   = [wb.word for wb in wbs]
        boxes   = [
            normalize_bbox(wb.x0, wb.y0, wb.x1, wb.y1, pw, ph)
            for wb in wbs
        ]
        page_inputs.append({
            "page_num":      pnum,
            "words":         words,
            "boxes":         boxes,
            "word_boxes_ref": wbs,
        })

    return page_inputs


# ─────────────────────────────────────────────────────────────
# Fallback: dummy bbox dari teks biasa (tanpa PDF)
# ─────────────────────────────────────────────────────────────
def text_to_words_and_boxes(text: str) -> Tuple[List[str], List[List[int]]]:
    """
    Fallback: buat words dan dummy bboxes dari teks polos.
    Setiap baris diberi bbox berdasarkan posisi baris (y) dan
    posisi karakter (x) dinormalisasi ke [0, 1000].

    Digunakan jika PDF tidak tersedia atau adalah scanned PDF.
    """
    lines  = text.split("\n")
    words: List[str]       = []
    boxes: List[List[int]] = []

    total_lines = max(len(lines), 1)

    for line_idx, line in enumerate(lines):
        tokens = line.split()
        if not tokens:
            continue

        # Y: posisi baris (0–1000)
        y0 = int(line_idx / total_lines * BBOX_NORM)
        y1 = int((line_idx + 1) / total_lines * BBOX_NORM)
        y1 = min(y1, BBOX_NORM)

        line_len = max(len(line), 1)
        char_pos = 0

        for tok in tokens:
            tok_start = line.find(tok, char_pos)
            tok_end   = tok_start + len(tok)

            x0 = int(tok_start / line_len * BBOX_NORM)
            x1 = int(tok_end   / line_len * BBOX_NORM)
            x1 = min(x1, BBOX_NORM)

            words.append(tok)
            boxes.append([x0, y0, x1, y1])
            char_pos = tok_end

    return words, boxes


# ─────────────────────────────────────────────────────────────
# ② LayoutLMv3 Inference
# ─────────────────────────────────────────────────────────────
def _predict_labels_layoutlm(
    words: List[str],
    boxes: List[List[int]],
    model_path: str = DEFAULT_LAYOUTLM_MODEL,
) -> Tuple[List[str], List[float]]:
    """
    Jalankan LayoutLMv3 pada satu halaman (words + normalized boxes).

    Returns:
        (labels, scores)
        labels : List[str] — prediksi label per kata
        scores : List[float] — confidence score per kata
    """
    import torch
    _load_layoutlm(model_path)

    processor = _layoutlm_processor
    model     = _layoutlm_model
    device    = _layoutlm_device

    # Potong jika terlalu panjang (LayoutLMv3 max 512 token)
    words = words[:LAYOUTLM_MAX_LEN]
    boxes = boxes[:LAYOUTLM_MAX_LEN]

    # Buat encoding — tanpa gambar (images=None) karena kita pakai text-only mode
    encoding = processor(
        text         = words,
        boxes        = boxes,
        is_split_into_words = True,
        return_tensors      = "pt",
        truncation          = True,
        max_length          = LAYOUTLM_MAX_LEN,
        padding             = "max_length",
    )

    # Pindahkan ke device
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    logits  = outputs.logits[0].cpu()         # (seq_len, num_labels)
    probs   = torch.softmax(logits, dim=-1)
    pred_ids = logits.argmax(dim=-1).numpy()
    scores   = probs.max(dim=-1).values.numpy()

    # id2label dari model config
    id2label = model.config.id2label

    # word_ids: mapping sub-token → kata asli
    word_ids = encoding.word_ids() if hasattr(encoding, "word_ids") else []
    if not word_ids:
        # Fallback: ambil dari input_ids secara langsung
        word_ids = encoding.get("word_ids", [None] * len(pred_ids))

    # Ambil prediksi per kata (hanya sub-token pertama)
    word_labels: Dict[int, str]   = {}
    word_scores: Dict[int, float] = {}
    prev_wid = None

    for idx, wid in enumerate(word_ids):
        if wid is None or wid == prev_wid:
            prev_wid = wid
            continue
        label = id2label.get(int(pred_ids[idx]), "O")
        word_labels[wid] = label
        word_scores[wid] = float(scores[idx])
        prev_wid = wid

    n_words = len(words)
    labels  = [word_labels.get(i, "O") for i in range(n_words)]
    sc      = [word_scores.get(i, 0.0) for i in range(n_words)]

    return labels, sc


# ─────────────────────────────────────────────────────────────
# ③ Span extraction dari prediksi label
# ─────────────────────────────────────────────────────────────
def _is_table_label(label: str) -> bool:
    """Cek apakah label merupakan label tabel (B-TABEL, I-TABEL, TABEL, dll.)."""
    label_upper = label.upper()
    return (
        label_upper in TABLE_LABEL_KEYWORDS
        or label_upper.endswith("-TABEL")
        or label_upper.endswith("-TABLE")
    )


def _extract_table_spans_from_page(
    words: List[str],
    labels: List[str],
    scores: List[float],
    word_boxes_ref: Optional[List[WordBox]] = None,
    page_num: int = 1,
    full_text: str = "",
) -> List[TableSpan]:
    """
    Konversi prediksi label per kata → List[TableSpan].

    Strategi:
    - Kumpulkan kata berturut-turut ber-label TABEL
    - Minimal MIN_TABLE_TOKENS kata → dianggap valid
    - Confidence = rata-rata score dalam span
    """
    spans: List[TableSpan] = []
    in_table    = False
    t_words:  List[str]   = []
    t_scores: List[float] = []
    t_start_idx: int      = 0

    def _flush(end_idx: int):
        nonlocal in_table, t_words, t_scores, t_start_idx
        if not in_table or len(t_words) < MIN_TABLE_TOKENS:
            in_table = False
            t_words  = []
            t_scores = []
            return

        span_text = " ".join(t_words)
        avg_conf  = sum(t_scores) / len(t_scores) if t_scores else 0.5

        # Coba temukan char offset di full_text
        char_start, char_end = 0, len(span_text)
        if full_text and span_text:
            idx = full_text.find(t_words[0])
            if idx >= 0:
                char_start = idx
                char_end   = idx + len(span_text)

        # Ambil char offset dari WordBox jika ada
        if word_boxes_ref and t_start_idx < len(word_boxes_ref):
            wb_start = word_boxes_ref[t_start_idx]
            char_start = wb_start.char_start
            wb_end_idx = min(end_idx - 1, len(word_boxes_ref) - 1)
            if wb_end_idx >= 0:
                char_end = word_boxes_ref[wb_end_idx].char_end

        spans.append(TableSpan(
            start      = char_start,
            end        = char_end,
            text       = span_text,
            page       = page_num,
            source     = "layoutlm",
            confidence = round(avg_conf, 4),
        ))

        in_table = False
        t_words  = []
        t_scores = []

    for i, (word, label, score) in enumerate(zip(words, labels, scores)):
        if _is_table_label(label):
            if not in_table:
                in_table    = True
                t_start_idx = i
            t_words.append(word)
            t_scores.append(score)
        else:
            _flush(i)

    _flush(len(words))
    return spans


# ─────────────────────────────────────────────────────────────
# Public API
# ─────────────────────────────────────────────────────────────
def detect_tables_from_pdf(
    pdf_path: str | Path,
    model_path: str = DEFAULT_LAYOUTLM_MODEL,
    min_confidence: float = 0.50,
) -> List[TableSpan]:
    """
    Deteksi tabel dalam PDF menggunakan LayoutLMv3.

    Alur:
      1. Ekstrak words + bboxes dari PDF (PyMuPDF)
      2. Proses per halaman dengan LayoutLMv3
      3. Kumpulkan span ber-label TABEL

    Args:
        pdf_path       : Path ke file PDF.
        model_path     : Model LayoutLMv3 (lokal atau HuggingFace Hub).
        min_confidence : Filter span di bawah threshold ini.

    Returns:
        List[TableSpan]
    """
    pdf_path = Path(pdf_path)

    try:
        word_boxes, page_count = extract_words_and_boxes(pdf_path)
    except Exception as e:
        logger.warning(f"[LayoutLM] Gagal ekstrak words dari PDF: {e}")
        return []

    if not word_boxes:
        logger.warning("[LayoutLM] Tidak ada kata yang diekstrak dari PDF.")
        return []

    page_inputs = build_page_inputs(word_boxes, pdf_path)
    all_spans: List[TableSpan] = []

    for page_inp in page_inputs:
        pnum   = page_inp["page_num"]
        words  = page_inp["words"]
        boxes  = page_inp["boxes"]
        wbrefs = page_inp["word_boxes_ref"]

        if not words:
            continue

        try:
            labels, scores = _predict_labels_layoutlm(words, boxes, model_path)
        except Exception as e:
            logger.warning(f"[LayoutLM] Prediksi halaman {pnum} gagal: {e}")
            continue

        spans = _extract_table_spans_from_page(
            words         = words,
            labels        = labels,
            scores        = scores,
            word_boxes_ref= wbrefs,
            page_num      = pnum,
        )
        all_spans.extend(spans)
        logger.debug(
            f"[LayoutLM] Hal. {pnum}: {len([l for l in labels if _is_table_label(l)])} "
            f"token TABEL → {len(spans)} span"
        )

    # Filter confidence
    filtered = [s for s in all_spans if s.confidence >= min_confidence]
    logger.info(
        f"[LayoutLM] Total: {len(all_spans)} span ditemukan, "
        f"{len(filtered)} lolos threshold confidence={min_confidence}"
    )
    return filtered


def detect_tables_from_text(
    text: str,
    model_path: str = DEFAULT_LAYOUTLM_MODEL,
    min_confidence: float = 0.50,
    page_num: int = 1,
) -> List[TableSpan]:
    """
    Deteksi tabel dari teks polos (fallback jika PDF tidak tersedia).
    Menggunakan dummy bounding boxes berbasis posisi baris.

    Args:
        text           : Teks dokumen lengkap.
        model_path     : Model LayoutLMv3.
        min_confidence : Threshold confidence.
        page_num       : Nomor halaman untuk metadata span.

    Returns:
        List[TableSpan]
    """
    words, boxes = text_to_words_and_boxes(text)
    if not words:
        return []

    try:
        labels, scores = _predict_labels_layoutlm(words, boxes, model_path)
    except Exception as e:
        logger.warning(f"[LayoutLM-Fallback] Prediksi gagal: {e}")
        return []

    spans = _extract_table_spans_from_page(
        words     = words,
        labels    = labels,
        scores    = scores,
        page_num  = page_num,
        full_text = text,
    )

    for s in spans:
        s.source = "layoutlm_fallback"

    filtered = [s for s in spans if s.confidence >= min_confidence]
    logger.info(
        f"[LayoutLM-Fallback] {len(filtered)} span tabel dari teks ({len(words)} kata)"
    )
    return filtered


def spans_to_text(spans: List[TableSpan], separator: str = "\n\n") -> str:
    """Gabungkan semua teks span tabel menjadi satu string."""
    return separator.join(s.text for s in spans if s.text.strip())


def spans_to_dict_list(spans: List[TableSpan]) -> List[Dict[str, Any]]:
    """Konversi list TableSpan ke list dict (untuk serialisasi JSON)."""
    return [s.to_dict() for s in spans]


# ─────────────────────────────────────────────────────────────
# Rule-based detector (tanpa model)
# ─────────────────────────────────────────────────────────────
def _rule_based_detect_tables(text: str, page_num: int = 1) -> List[TableSpan]:
    """
    Deteksi tabel dari teks menggunakan pola regex.
    Lebih cepat dan tidak butuh GPU — digunakan sebagai pre-screening
    sebelum LayoutLMv3 dan sebagai sole detector jika transformer tidak tersedia.

    Returns:
        List[TableSpan] dengan source="rule_based"
    """
    lines   = text.split("\n")
    blocks: List[TableSpan] = []
    in_block        = False
    block_lines: List[str] = []
    block_start_char = 0
    char_offset      = 0
    consecutive_miss = 0
    MAX_MISS         = 3

    def _is_table_line(line: str, in_blk: bool) -> bool:
        return (
            bool(_RULE_PIPE_ROW.search(line))
            or bool(_RULE_SEPARATOR.search(line))
            or bool(_RULE_TABLE_HEADER.search(line))
            or bool(_RULE_AMOUNT_ROW.search(line))
            or (in_blk and bool(_re.match(r'^\s*\d+[.)\s]', line)))
            # Spasi-kolom hanya jika baris cukup panjang (hindari false positive)
            or (len(line) > 20 and bool(_RULE_SPACE_COL.search(line)))
        )

    def _flush() -> None:
        nonlocal in_block, block_lines, block_start_char, consecutive_miss
        useful = [l for l in block_lines if l.strip()]
        if len(useful) >= 3:
            span_text = "\n".join(block_lines).strip()
            end_char  = block_start_char + len(span_text)
            blocks.append(TableSpan(
                start      = block_start_char,
                end        = end_char,
                text       = span_text,
                page       = page_num,
                source     = "rule_based",
                confidence = 0.75,   # fixed confidence untuk rule-based
            ))
        in_block        = False
        block_lines     = []
        consecutive_miss = 0

    for line in lines:
        if _is_table_line(line, in_block):
            if not in_block:
                in_block         = True
                block_start_char = char_offset
                block_lines      = []
            block_lines.append(line)
            consecutive_miss = 0
        elif in_block:
            consecutive_miss += 1
            block_lines.append(line)
            if consecutive_miss >= MAX_MISS:
                _flush()
        char_offset += len(line) + 1  # +1 for "\n"

    if in_block:
        _flush()

    logger.debug(f"[RuleBased] {len(blocks)} blok tabel dari {len(lines)} baris teks")
    return blocks


# ─────────────────────────────────────────────────────────────
# Backward-compat alias (dipanggil dari inference.py)
# ─────────────────────────────────────────────────────────────
def hybrid_detect(
    text: str,
    use_transformer: bool = True,
    min_confidence: float = 0.50,
    pdf_path: Optional[str] = None,
    model_path: str = DEFAULT_LAYOUTLM_MODEL,
) -> List[TableSpan]:
    """
    Entry point utama — Hybrid Rule-Based + Transformer.

    Alur:
    1. Rule-based: deteksi cepat via regex (selalu dijalankan)
    2. Transformer (LayoutLMv3): lebih akurat, dijalankan jika
       use_transformer=True dan teks >= 50 kata.
    3. Merge hasil: gabungkan span rule-based + transformer,
       hindari duplikasi (overlap > 50%).

    Jika pdf_path diberikan → PDF path dipakai untuk LayoutLMv3
    (bbox asli lebih akurat daripada dummy bbox dari teks).

    Args:
        text           : Teks dokumen.
        use_transformer: Aktifkan LayoutLMv3 (default True).
        min_confidence : Threshold confidence untuk filter span.
        pdf_path       : Path PDF opsional untuk LayoutLMv3.
        model_path     : Model LayoutLMv3.

    Returns:
        List[TableSpan]
    """
    all_spans: List[TableSpan] = []

    # ── Tahap 1: Rule-based (selalu dijalankan) ────────────────
    rule_spans = _rule_based_detect_tables(text, page_num=1)
    all_spans.extend(rule_spans)
    logger.info(f"[Hybrid] Rule-based: {len(rule_spans)} span tabel ditemukan")

    # ── Tahap 2: Transformer (opsional) ───────────────────────
    word_count = len(text.split()) if text else 0
    if use_transformer and word_count >= 20:
        try:
            if pdf_path and Path(pdf_path).exists():
                logger.info("[Hybrid] LayoutLMv3 dari PDF path")
                transformer_spans = detect_tables_from_pdf(
                    pdf_path       = pdf_path,
                    model_path     = model_path,
                    min_confidence = min_confidence,
                )
            else:
                logger.info("[Hybrid] LayoutLMv3 dari teks (dummy bbox)")
                transformer_spans = detect_tables_from_text(
                    text           = text,
                    model_path     = model_path,
                    min_confidence = min_confidence,
                )
            logger.info(f"[Hybrid] LayoutLMv3: {len(transformer_spans)} span tabel")
            all_spans.extend(transformer_spans)
        except Exception as e:
            logger.warning(f"[Hybrid] LayoutLMv3 gagal (fallback ke rule-based saja): {e}")
    elif not use_transformer:
        logger.info("[Hybrid] Transformer dinonaktifkan, hanya rule-based")
    else:
        logger.info(f"[Hybrid] Teks terlalu pendek ({word_count} kata), skip transformer")

    # ── Tahap 3: Merge & deduplikasi ──────────────────────────
    if len(all_spans) <= 1:
        return [s for s in all_spans if s.confidence >= min_confidence]

    # Sort by start position
    all_spans.sort(key=lambda s: s.start)

    merged: List[TableSpan] = []
    for span in all_spans:
        if not merged:
            merged.append(span)
            continue
        prev = merged[-1]
        # Hitung overlap
        overlap_start = max(span.start, prev.start)
        overlap_end   = min(span.end,   prev.end)
        overlap_len   = max(0, overlap_end - overlap_start)
        prev_len      = max(1, prev.end - prev.start)
        span_len      = max(1, span.end - span.start)
        overlap_ratio = overlap_len / min(prev_len, span_len)

        if overlap_ratio > 0.5:
            # Overlap signifikan — ambil span dengan confidence lebih tinggi
            # atau gabung teksnya jika dari sumber berbeda
            if span.source != prev.source:
                # Gabungkan: perluas span, gabung teks, ambil confidence max
                new_start = min(prev.start, span.start)
                new_end   = max(prev.end,   span.end)
                new_text  = prev.text if len(prev.text) >= len(span.text) else span.text
                merged[-1] = TableSpan(
                    start      = new_start,
                    end        = new_end,
                    text       = new_text,
                    page       = prev.page,
                    source     = "hybrid",
                    confidence = max(prev.confidence, span.confidence),
                )
            elif span.confidence > prev.confidence:
                merged[-1] = span
            # else: pertahankan prev
        else:
            merged.append(span)

    filtered = [s for s in merged if s.confidence >= min_confidence]
    logger.info(
        f"[Hybrid] Final: {len(merged)} span setelah merge, "
        f"{len(filtered)} lolos threshold confidence={min_confidence}"
    )
    return filtered



### Menulis file `src/finetune_indobert.py`

In [ ]:
%%writefile src/finetune_indobert.py
from __future__ import annotations

import argparse
import json
import math
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from loguru import logger
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
    get_cosine_with_hard_restarts_schedule_with_warmup,
)
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

try:
    from datasets import load_dataset
    HAS_DATASETS = True
except ImportError:
    HAS_DATASETS = False
    logger.warning("datasets library not found. HF dataset mode will be disabled.")

sys.path.insert(0, str(Path(__file__).parent.parent))
from config import CKPT_DIR, ID2LABEL, LABEL2ID, NUM_LABELS, LABELS, model_cfg
from src.dataset import (
    NERDataset,
    build_training_samples,
    get_tokenizer,
    load_json_dataset,
    split_dataset,
)
from src.model import build_model, get_device


# ══════════════════════════════════════════════════════════════
# ① LABEL SMOOTHING LOSS
# ══════════════════════════════════════════════════════════════
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy dengan label smoothing.
    Distribusi target: (1 - ε) pada kelas benar, ε/(K-1) pada sisanya.
    Special token (-100) otomatis diabaikan.
    """
    def __init__(self, smoothing: float = 0.1, ignore_index: int = -100):
        super().__init__()
        self.smoothing    = smoothing
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_cls = logits.size(-1)
        flat_logits  = logits.view(-1, n_cls)
        flat_targets = targets.view(-1)

        mask = flat_targets != self.ignore_index
        flat_logits  = flat_logits[mask]
        flat_targets = flat_targets[mask]

        if flat_logits.numel() == 0:
            return flat_logits.sum() * 0.0

        log_probs = F.log_softmax(flat_logits, dim=-1)
        nll       = F.nll_loss(log_probs, flat_targets, reduction="mean")
        smooth    = -log_probs.mean()
        return (1.0 - self.smoothing) * nll + self.smoothing * smooth


# ══════════════════════════════════════════════════════════════
# ② LAYER-WISE LEARNING RATE DECAY
# ══════════════════════════════════════════════════════════════
def build_llrd_param_groups(
    model: nn.Module,
    base_lr: float       = 2e-5,
    classifier_lr: float = 5e-5,
    decay: float         = 0.9,
    weight_decay: float  = 0.01,
) -> List[Dict]:
    """
    Buat parameter groups dengan LR berbeda per layer BERT.

    Hierarki LR (dari paling tinggi ke rendah):
        classifier head  → classifier_lr           (layer paling atas)
        encoder layer N  → base_lr * decay^0       (layer BERT teratas)
        encoder layer N-1→ base_lr * decay^1
        ...
        encoder layer 0  → base_lr * decay^N       (layer BERT terbawah)
        embedding layer  → base_lr * decay^(N+1)   (paling kecil)
    """
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}

    def _group(params, lr):
        decay_p  = [p for n, p in params if not any(nd in n for nd in no_decay)]
        nodecay_p = [p for n, p in params if any(nd in n for nd in no_decay)]
        groups = []
        if decay_p:
            groups.append({"params": decay_p,  "lr": lr, "weight_decay": weight_decay})
        if nodecay_p:
            groups.append({"params": nodecay_p, "lr": lr, "weight_decay": 0.0})
        return groups

    param_groups = []

    # ── Classifier head (highest LR) ──────────────────────────
    classifier_params = [
        (n, p) for n, p in model.named_parameters()
        if "classifier" in n or "pooler" in n
    ]
    param_groups.extend(_group(classifier_params, classifier_lr))

    # ── Encoder layers (LLRD) ─────────────────────────────────
    encoder_layers = list(model.bert.encoder.layer)
    n_layers = len(encoder_layers)

    for i, layer_module in enumerate(reversed(encoder_layers)):
        # i=0 → topmost layer (layer N), i=N-1 → bottom layer (layer 0)
        layer_lr = base_lr * (decay ** i)
        layer_params = [
            (f"encoder.layer.{n_layers-1-i}.{n}", p)
            for n, p in layer_module.named_parameters()
        ]
        param_groups.extend(_group(layer_params, layer_lr))

    # ── Embeddings (lowest LR) ────────────────────────────────
    emb_lr = base_lr * (decay ** n_layers)
    emb_params = list(model.bert.embeddings.named_parameters())
    param_groups.extend(_group(emb_params, emb_lr))

    # Log distribusi LR
    lrs = sorted(set(g["lr"] for g in param_groups), reverse=True)
    logger.info(f"LLRD: {len(lrs)} level LR — max={max(lrs):.2e}, min={min(lrs):.2e}")

    return param_groups


# ══════════════════════════════════════════════════════════════
# ③ PROGRESSIVE UNFREEZING CALLBACK
# ══════════════════════════════════════════════════════════════
class ProgressiveUnfreezingCallback(TrainerCallback):
    """
    Callback HuggingFace Trainer untuk Progressive Unfreezing.

    Jadwal:
      Epoch 0             : hanya classifier head yang trainable
      Setiap unfreeze_every epoch : buka unfreeze_n layer dari atas ke bawah
      Setelah semua layer terbuka : training normal penuh
    """

    def __init__(
        self,
        model: nn.Module,
        total_layers: int   = 12,
        unfreeze_n: int     = 2,
        unfreeze_every: int = 2,
    ):
        self.model          = model
        self.total_layers   = total_layers
        self.unfreeze_n     = unfreeze_n
        self.unfreeze_every = unfreeze_every
        self.unfrozen_up_to = -1  # berapa layer (dari atas) yang sudah dibuka

        # Mulai: freeze semua kecuali classifier
        self._freeze_all_bert()
        logger.info("Progressive Unfreezing: mulai dengan hanya classifier head trainable.")

    def _freeze_all_bert(self):
        for param in self.model.bert.parameters():
            param.requires_grad = False

    def _unfreeze_layer(self, layer_idx: int):
        """Unfreeze satu encoder layer (diindex dari atas=0)."""
        actual_idx = self.total_layers - 1 - layer_idx
        if actual_idx < 0:
            return
        for param in self.model.bert.encoder.layer[actual_idx].parameters():
            param.requires_grad = True

    def _unfreeze_embeddings(self):
        for param in self.model.bert.embeddings.parameters():
            param.requires_grad = True

    def on_epoch_begin(
        self, args, state: TrainerState, control: TrainerControl, **kwargs
    ):
        epoch = int(state.epoch) if state.epoch else 0
        if epoch == 0:
            return

        # Hitung berapa layer yang seharusnya terbuka sekarang
        target_unfrozen = min(
            (epoch // self.unfreeze_every) * self.unfreeze_n,
            self.total_layers,
        )

        newly_opened = []
        while self.unfrozen_up_to + 1 < target_unfrozen:
            self.unfrozen_up_to += 1
            self._unfreeze_layer(self.unfrozen_up_to)
            newly_opened.append(self.total_layers - 1 - self.unfrozen_up_to)

        if newly_opened:
            trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            logger.info(
                f"[Epoch {epoch}] Membuka encoder layer: {newly_opened} | "
                f"Trainable params: {trainable:,}"
            )

        # Buka embeddings setelah semua layer encoder terbuka
        if self.unfrozen_up_to >= self.total_layers - 1:
            self._unfreeze_embeddings()


# ══════════════════════════════════════════════════════════════
# ④ CUSTOM TRAINER (Label Smoothing + LLRD optimizer)
# ══════════════════════════════════════════════════════════════
class NERTrainer(Trainer):
    """
    Ekstensi HuggingFace Trainer dengan:
    - Label smoothing loss
    - LLRD optimizer (opsional)
    """

    def __init__(
        self,
        label_smoothing: float = 0.0,
        use_llrd: bool         = False,
        llrd_decay: float      = 0.9,
        classifier_lr: float   = 5e-5,
        *args, **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.label_smoothing = label_smoothing
        self.use_llrd        = use_llrd
        self.llrd_decay      = llrd_decay
        self.classifier_lr   = classifier_lr

        if label_smoothing > 0:
            self._loss_fn = LabelSmoothingCrossEntropy(smoothing=label_smoothing)
            logger.info(f"Label Smoothing aktif: ε={label_smoothing}")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        if self.label_smoothing > 0:
            loss = self._loss_fn(logits, labels)
        else:
            loss = nn.CrossEntropyLoss(ignore_index=-100)(
                logits.view(-1, NUM_LABELS), labels.view(-1)
            )

        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        """Override optimizer dengan LLRD jika aktif."""
        if not self.use_llrd:
            return super().create_optimizer()

        param_groups = build_llrd_param_groups(
            model        = self.model,
            base_lr      = self.args.learning_rate,
            classifier_lr= self.classifier_lr,
            decay        = self.llrd_decay,
            weight_decay = self.args.weight_decay,
        )
        self.optimizer = AdamW(
            param_groups,
            eps   = self.args.adam_epsilon,
            betas = (self.args.adam_beta1, self.args.adam_beta2),
        )
        return self.optimizer


# ══════════════════════════════════════════════════════════════
# ⑤ COMPUTE METRICS untuk Trainer
# ══════════════════════════════════════════════════════════════
def make_compute_metrics():
    """Closure yang mengembalikan fungsi compute_metrics untuk Trainer."""

    def compute_metrics(eval_pred):
        try:
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)

            true_seqs, pred_seqs = [], []
            for pred_row, label_row in zip(predictions, labels):
                t_seq, p_seq = [], []
                for p, l in zip(pred_row, label_row):
                    if l == -100:
                        continue
                    t_seq.append(ID2LABEL.get(int(l), "O"))
                    p_seq.append(ID2LABEL.get(int(p), "O"))
                if t_seq:
                    true_seqs.append(t_seq)
                    pred_seqs.append(p_seq)

            if not true_seqs:
                return {"f1": 0.0, "precision": 0.0, "recall": 0.0}

            return {
                "f1":        f1_score(true_seqs, pred_seqs),
                "precision": precision_score(true_seqs, pred_seqs),
                "recall":    recall_score(true_seqs, pred_seqs),
            }
        except Exception as e:
            logger.error(f"Error computing metrics: {e}")
            return {"f1": 0.0, "precision": 0.0, "recall": 0.0}

    return compute_metrics





# ══════════════════════════════════════════════════════════════
# ⑥ MAIN FINE-TUNE FUNCTION
# ══════════════════════════════════════════════════════════════
def finetune_indobert(
    dataset_path: str,
    strategy: str           = "full",
    output_dir: str         = str(CKPT_DIR / "indobert-ner-finetuned"),
    epochs: int             = 10,
    batch_size: int         = 8,
    learning_rate: float    = 2e-5,
    classifier_lr: float    = 5e-5,
    weight_decay: float     = 0.01,
    warmup_ratio: float     = 0.1,
    gradient_accumulation: int = 2,
    label_smoothing: float  = 0.0,
    llrd_decay: float       = 0.9,
    unfreeze_n: int         = 2,
    unfreeze_every: int     = 2,
    fp16: bool              = True,
    patience: int           = 3,
    combine_hf: bool        = False,
    seed: int               = 42,
):
    """
    Fine-tune IndoBERT dengan strategi yang dipilih.

    Args:
        strategy : Pilih salah satu:
            "full"        → fine-tune semua layer, LR seragam
            "llrd"        → fine-tune semua layer, LR decay per layer
            "progressive" → unfreeze layer secara bertahap per epoch
        label_smoothing : 0.0 = off; 0.1 = recommended
        gradient_accumulation: batch efektif = batch_size * gradient_accumulation
    """
    torch.manual_seed(seed)
    logger.info(f"\n{'═'*60}")
    logger.info(f"  Fine-tune IndoBERT | Strategi: {strategy.upper()}")
    if combine_hf:
        logger.info(f"  Data: HuggingFace + Local")
    logger.info(f"{'═'*60}")

    # ── GPU Detection ─────────────────────────────────────────
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"  🖥️  GPU         : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
        logger.info(f"  ⚡ CUDA        : {torch.version.cuda}")
        logger.info(f"  🔄 FP16        : {'Aktif ✓' if fp16 else 'Non-aktif'}")
    else:
        logger.warning("  ⚠️  GPU tidak terdeteksi — training akan berjalan di CPU")
        fp16 = False  # FP16 hanya untuk GPU

    effective_batch = batch_size * gradient_accumulation
    logger.info(f"  LR            : {learning_rate:.2e}")
    logger.info(f"  Classifier LR : {classifier_lr:.2e}")
    logger.info(f"  Epochs        : {epochs}")
    logger.info(f"  Batch size    : {batch_size} × accum {gradient_accumulation} = {effective_batch} efektif")
    logger.info(f"  Label smooth  : {label_smoothing}")
    logger.info(f"  LLRD decay    : {llrd_decay if strategy=='llrd' else 'N/A'}")
    logger.info(f"{'═'*60}\n")

    # ── Load & prep data ──────────────────────────────────────
    # Strategy: Gunakan HF dataset + local jika dipilih
    if combine_hf:
        try:
            from src.dataset import load_dataset_combined
            combined_samp = load_dataset_combined(
                hf_only=False,
                local_path=dataset_path,
                hf_split="train"
            )
            logger.info(f"Combined: HF + local total {len(combined_samp)} samples")
        except Exception as e:
            logger.error(f"Gagal combine HF + local: {e}")
            raise
    else:
        # Default: gunakan local dataset saja
        logger.info(f"Hanya menggunakan dataset lokal: {dataset_path}")
        local_raw = load_json_dataset(dataset_path)
        combined_samp = build_training_samples(local_raw)
    
    # Split: train/val/test dari combined dataset
    train_data, val_data, test_data = split_dataset(combined_samp, seed=seed)
    
    logger.info(f"  Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

    train_ds = NERDataset(train_data)
    val_ds   = NERDataset(val_data)

    # ── Build model ───────────────────────────────────────────
    model = build_model()

    # ── Callbacks ─────────────────────────────────────────────
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience  = patience,
            early_stopping_threshold = 1e-4,
        )
    ]

    if strategy == "progressive":
        n_bert_layers = len(model.bert.encoder.layer)
        prog_cb = ProgressiveUnfreezingCallback(
            model         = model,
            total_layers  = n_bert_layers,
            unfreeze_n    = unfreeze_n,
            unfreeze_every= unfreeze_every,
        )
        callbacks.append(prog_cb)
        logger.info(
            f"Progressive Unfreezing: {n_bert_layers} layers, "
            f"buka {unfreeze_n} layer setiap {unfreeze_every} epoch"
        )

    # ── Training Arguments ────────────────────────────────────
    use_fp16 = fp16 and torch.cuda.is_available()
    training_args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size * 2,
        gradient_accumulation_steps = gradient_accumulation,
        learning_rate               = learning_rate,
        weight_decay                = weight_decay,
        warmup_ratio                = warmup_ratio,
        fp16                        = use_fp16,
        use_cpu                     = False,      # Force GPU jika tersedia
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_f1",
        greater_is_better           = True,
        logging_dir                 = str(Path(output_dir) / "logs"),
        logging_strategy            = "epoch",
        save_total_limit            = 2,
        seed                        = seed,
        report_to                   = "none",    # ganti ke "wandb" jika mau
        adam_epsilon                = 1e-8,
        adam_beta1                  = 0.9,
        adam_beta2                  = 0.999,
        max_grad_norm               = 1.0,
        dataloader_num_workers      = 0,
        label_names                 = ["labels"],
    )

    # ── Konfirmasi device ─────────────────────────────────────
    logger.info(f"  🎯 Training device: {training_args.device}")
    logger.info(f"  🎯 n_gpu: {training_args.n_gpu}")
    if str(training_args.device) == "cpu":
        logger.warning("⚠️  Training berjalan di CPU! Cek instalasi CUDA.")

    # ── Trainer ───────────────────────────────────────────────
    trainer = NERTrainer(
        model             = model,
        args              = training_args,
        train_dataset     = train_ds,
        eval_dataset      = val_ds,
        compute_metrics   = make_compute_metrics(),
        callbacks         = callbacks,
        label_smoothing   = label_smoothing,
        use_llrd          = (strategy == "llrd"),
        llrd_decay        = llrd_decay,
        classifier_lr     = classifier_lr,
    )

    # ── Train ─────────────────────────────────────────────────
    logger.info("Mulai training...\n")
    train_result = trainer.train()

    # ── Simpan model & tokenizer terbaik ─────────────────────
    trainer.save_model(output_dir)
    get_tokenizer().save_pretrained(output_dir)

    # ── Evaluasi test set ─────────────────────────────────────
    if test_data:
        test_ds = NERDataset(test_data)
        logger.info("\nEvaluasi Test Set...")
        test_result = trainer.evaluate(test_ds, metric_key_prefix="test")
        logger.info(
            f"Test F1={test_result['test_f1']:.4f} | "
            f"P={test_result['test_precision']:.4f} | "
            f"R={test_result['test_recall']:.4f}"
        )
    else:
        test_result = {}

    # ── Simpan laporan ────────────────────────────────────────
    report = {
        "strategy":          strategy,
        "hyperparams": {
            "learning_rate":        learning_rate,
            "classifier_lr":        classifier_lr,
            "epochs":               epochs,
            "batch_size":           batch_size,
            "gradient_accumulation":gradient_accumulation,
            "label_smoothing":      label_smoothing,
            "llrd_decay":           llrd_decay,
            "warmup_ratio":         warmup_ratio,
        },
        "train_metrics": train_result.metrics,
        "test_metrics":  test_result,
    }

    report_path = Path(output_dir) / "finetune_report.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    # ── Print Summary ─────────────────────────────────────────
    logger.info("\n" + "="*70)
    logger.info("  TRAINING SUMMARY")
    logger.info("="*70)
    logger.info(f"  Strategy        : {strategy.upper()}")
    logger.info(f"  Epochs          : {epochs}")
    logger.info(f"  Train Loss      : {train_result.metrics.get('train_loss', 0):.4f}")
    logger.info(f"  Train Samples   : {len(train_data)}")
    logger.info(f"  Val Samples     : {len(val_data)}")
    logger.info(f"  Test Samples    : {len(test_data)}")
    
    if test_result:
        logger.info("\n  TEST RESULTS:")
        logger.info(f"    F1 Score      : {test_result.get('test_f1', 0):.4f}")
        logger.info(f"    Precision     : {test_result.get('test_precision', 0):.4f}")
        logger.info(f"    Recall        : {test_result.get('test_recall', 0):.4f}")
        logger.info(f"    Loss          : {test_result.get('test_loss', 0):.4f}")
    
    logger.info(f"\n  Model           : {output_dir}")
    logger.info(f"  Report          : {report_path}")
    logger.info("="*70 + "\n")

    return report


# ══════════════════════════════════════════════════════════════
# CLI
# ══════════════════════════════════════════════════════════════
def parse_args():
    p = argparse.ArgumentParser(description="Fine-tuning IndoBERT NER")
    p.add_argument("--dataset",      required=True,
                   help="Path ke local dataset JSON")
    p.add_argument("--combine-hf",   action="store_true",
                   help="Combine HF indo-ner-dataset dengan local data untuk training")
    p.add_argument("--output",       default=str(CKPT_DIR / "indobert-ner-finetuned"))
    p.add_argument("--strategy",     default="full",
                   choices=["full", "llrd", "progressive"],
                   help="full: LR seragam | llrd: decay per layer | progressive: unfreeze bertahap")
    p.add_argument("--epochs",       type=int,   default=10)
    p.add_argument("--batch",        type=int,   default=8,
                   help="Batch size per device (default 8 untuk RTX 3060 6GB)")
    p.add_argument("--lr",           type=float, default=2e-5)
    p.add_argument("--clf-lr",       type=float, default=5e-5,
                   help="LR khusus classifier head (untuk llrd)")
    p.add_argument("--weight-decay", type=float, default=0.01)
    p.add_argument("--warmup",       type=float, default=0.1)
    p.add_argument("--accum",        type=int,   default=2,
                   help="Gradient accumulation steps (default 2 → effective batch=16)")
    p.add_argument("--smoothing",    type=float, default=0.0,
                   help="Label smoothing epsilon (0.0=off, 0.1=recommended)")
    p.add_argument("--llrd-decay",   type=float, default=0.9,
                   help="Faktor decay LLRD per layer (0.8–0.95)")
    p.add_argument("--unfreeze-n",   type=int,   default=2,
                   help="Jumlah layer yang dibuka per tahap (progressive)")
    p.add_argument("--unfreeze-every",type=int,  default=2,
                   help="Buka layer setiap N epoch (progressive)")
    p.add_argument("--patience",     type=int,   default=3,
                   help="Early stopping patience")
    p.add_argument("--fp16",         action="store_true", default=True,
                   help="Gunakan Mixed Precision FP16 (default: aktif untuk GPU)")
    p.add_argument("--no-fp16",      action="store_true",
                   help="Matikan FP16 (fallback ke FP32)")
    p.add_argument("--seed",         type=int,   default=42)
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    # --no-fp16 override
    use_fp16 = args.fp16 and not args.no_fp16
    finetune_indobert(
        dataset_path        = args.dataset,
        strategy            = args.strategy,
        output_dir          = args.output,
        epochs              = args.epochs,
        batch_size          = args.batch,
        learning_rate       = args.lr,
        classifier_lr       = args.clf_lr,
        weight_decay        = args.weight_decay,
        warmup_ratio        = args.warmup,
        gradient_accumulation = args.accum,
        label_smoothing     = args.smoothing,
        llrd_decay          = args.llrd_decay,
        unfreeze_n          = args.unfreeze_n,
        unfreeze_every      = args.unfreeze_every,
        fp16                = use_fp16,
        patience            = args.patience,
        combine_hf          = args.combine_hf,
        seed                = args.seed,
    )


### Menulis file `src/inference.py`

In [ ]:
%%writefile src/inference.py
"""
src/inference.py
Inference engine: teks dokumen → NER → entity dict → JSON ringkasan.

Fungsi utama:
    run_ner(text, pdf_path=None) -> dict   # ekstrak entitas dari teks
    run_ner_batch(texts)         -> list
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import spacy
import torch
from loguru import logger
from transformers import AutoModelForTokenClassification, AutoTokenizer

from config import ID2LABEL, LABEL2ID, LABELS, model_cfg


# ─────────────────────────────────────────────────────────────
# Model & tokenizer loading (singleton)
# ─────────────────────────────────────────────────────────────
_model: Optional[AutoModelForTokenClassification] = None
_tokenizer: Optional[AutoTokenizer]               = None
_device: Optional[torch.device]                   = None


def load_model(checkpoint_path: Optional[str] = None) -> None:
    """Load model dan tokenizer ke memori. Panggil sekali saja."""
    global _model, _tokenizer, _device

    path = checkpoint_path or model_cfg.fine_tuned_path

    from src.model import get_device
    _device = get_device()

    logger.info(f"Memuat model inferensi dari: {path}")
    _tokenizer = AutoTokenizer.from_pretrained(path)
    _model     = AutoModelForTokenClassification.from_pretrained(
        path,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(_device)
    _model.eval()
    logger.info("Model siap untuk inferensi.")


def _ensure_model():
    if _model is None or _tokenizer is None:
        load_model()


# ─────────────────────────────────────────────────────────────
# spaCy pre-tokenizer (sentence & word splitting)
# ─────────────────────────────────────────────────────────────
_nlp: Optional[spacy.Language] = None

def _get_nlp():
    global _nlp
    if _nlp is None:
        try:
            _nlp = spacy.load("xx_ent_wiki_sm")
        except OSError:
            _nlp = spacy.blank("id")
        
        # Add sentencizer if not already in pipeline
        if "sentencizer" not in _nlp.pipe_names:
            _nlp.add_pipe("sentencizer")
    return _nlp


# ─────────────────────────────────────────────────────────────
# Core NER inference (sliding window untuk teks panjang)
# ─────────────────────────────────────────────────────────────
def _predict_tokens(
    tokens: List[str],
    stride: int  = 64,
    max_len: int = 512,
) -> List[str]:
    """
    Jalankan model pada token list, dengan sliding window untuk
    teks melebihi max_len sub-word token.

    Returns:
        List[str] BIO tag per token input (satu tag per token kata, bukan sub-word).
    """
    _ensure_model()
    tokenizer = _tokenizer
    model     = _model
    device    = _device

    # Tokenisasi dengan is_split_into_words=True
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_offsets_mapping=False,
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt",
    )

    word_ids = encoding.word_ids()

    with torch.no_grad():
        outputs  = model(
            input_ids      = encoding["input_ids"].to(device),
            attention_mask = encoding["attention_mask"].to(device),
        )
    logits  = outputs.logits[0].cpu()                       # (seq_len, num_labels)
    pred_ids = logits.argmax(dim=-1).numpy()

    # Ambil satu prediksi per kata (token pertama sub-word)
    word_preds: Dict[int, str] = {}
    prev_wid = None
    for idx, wid in enumerate(word_ids):
        if wid is None or wid == prev_wid:
            prev_wid = wid
            continue
        word_preds[wid] = ID2LABEL.get(int(pred_ids[idx]), "O")
        prev_wid = wid

    return [word_preds.get(i, "O") for i in range(len(tokens))]


# ─────────────────────────────────────────────────────────────
# Entity span extraction dari BIO sequence
# ─────────────────────────────────────────────────────────────
def extract_spans(
    tokens: List[str],
    bio_tags: List[str],
) -> List[Dict[str, Any]]:
    """
    Konversi BIO token-tag sequence → list entitas.

    Returns:
        [{"label": str, "value": str, "tokens": List[str]}, ...]
    """
    spans, current_label, current_tokens = [], None, []

    for token, tag in zip(tokens, bio_tags):
        if tag.startswith("B-"):
            if current_label and current_tokens:
                spans.append({"label": current_label, "value": " ".join(current_tokens)})
            current_label  = tag[2:]
            current_tokens = [token]
        elif tag.startswith("I-") and current_label:
            label = tag[2:]
            if label == current_label:
                current_tokens.append(token)
            else:
                # Label berbeda → flush & mulai baru
                spans.append({"label": current_label, "value": " ".join(current_tokens)})
                current_label  = label
                current_tokens = [token]
        else:
            if current_label and current_tokens:
                spans.append({"label": current_label, "value": " ".join(current_tokens)})
            current_label, current_tokens = None, []

    if current_label and current_tokens:
        spans.append({"label": current_label, "value": " ".join(current_tokens)})

    return spans


# ─────────────────────────────────────────────────────────────
# Aggregate spans → structured dict
# ─────────────────────────────────────────────────────────────
def aggregate_entities(spans: List[Dict]) -> Dict[str, Any]:
    """
    Kumpulkan semua span per label. Jika satu label muncul beberapa kali,
    simpan sebagai list; jika hanya sekali, simpan sebagai string.
    """
    result: Dict[str, Any] = {label: None for label in LABELS}
    bucket: Dict[str, List[str]] = {label: [] for label in LABELS}

    for span in spans:
        lbl = span["label"]
        if lbl in bucket:
            bucket[lbl].append(span["value"])

    for lbl, values in bucket.items():
        if len(values) == 0:
            result[lbl] = None
        elif len(values) == 1:
            result[lbl] = values[0]
        else:
            result[lbl] = values     # beberapa kemunculan

    return result


# ─────────────────────────────────────────────────────────────
# Post-processing helpers
# ─────────────────────────────────────────────────────────────
def _extract_perihal_from_text(text: str) -> Optional[str]:
    """
    Ekstrak PERIHAL dari dokumen dengan mencari pola 'Perihal :', 'Hal :', dll.
    Ini lebih reliable daripada NER model untuk field ini.
    """
    match = re.search(r'(?:perihal|hal|topik|subject)\s*:\s*([^\n]+)', text, re.IGNORECASE)
    if match:
        perihal_text = match.group(1).strip()
        perihal_text = re.sub(r'\s+', ' ', perihal_text)
        perihal_text = perihal_text.strip()
        if perihal_text and len(perihal_text) > 2:
            return perihal_text
    return None


def _extract_isi_from_text(text: str, max_chars: int = 800) -> Optional[str]:
    """
    Ekstrak isi utama dokumen secara rule-based.
    Cari paragraf setelah baris 'Perihal' atau setelah pembuka surat,
    kecualikan baris header/metadata.
    """
    lines = text.split("\n")

    # Temukan posisi setelah baris 'Perihal'
    perihal_idx = -1
    for i, line in enumerate(lines):
        if re.search(r'(?:perihal|hal|topik|subject)\s*:', line, re.IGNORECASE):
            perihal_idx = i
            break

    start_idx = perihal_idx + 1 if perihal_idx >= 0 else 0

    # Pola kata pembuka surat umum
    _OPENING_WORDS = re.compile(
        r'^(?:dengan\s+hormat|bersama\s+ini|sehubungan\s+dengan|'
        r'menindaklanjuti|berkenaan\s+dengan|dalam\s+rangka|'
        r'berdasarkan|yang\s+bertanda\s+tangan|menerangkan\s+bahwa|'
        r'sesuai\s+dengan|diberitahukan\s+bahwa)',
        re.IGNORECASE,
    )

    paragraphs = []
    for line in lines[start_idx:]:
        stripped = line.strip()
        if not stripped:
            continue
        # Lewati baris metadata/header
        if re.match(
            r'^(nomor|nip|kepada|dari|perihal|hal|tanggal|lampiran)\s*:',
            stripped, re.IGNORECASE
        ):
            continue
        # Ambil baris yang merupakan paragraf isi
        if len(stripped) > 20 or _OPENING_WORDS.match(stripped):
            paragraphs.append(stripped)
        if sum(len(p) for p in paragraphs) >= max_chars:
            break

    if not paragraphs:
        return None

    isi = " ".join(paragraphs)
    isi = re.sub(r'\s+', ' ', isi).strip()
    return isi[:max_chars] if len(isi) > max_chars else isi


def _extract_table_content(
    text: str,
    pdf_path: Optional[str] = None,
    model_path: Optional[str] = None,
) -> Optional[str]:
    """
    Deteksi dan ekstrak konten tabel menggunakan LayoutLMv3.

    Args:
        text       : Teks dokumen (fallback jika pdf_path tidak ada).
        pdf_path   : Path ke file PDF untuk hasil lebih akurat.
        model_path : Model LayoutLMv3 (default: microsoft/layoutlmv3-base).

    Returns:
        String gabungan semua blok tabel yang ditemukan, atau None.
    """
    try:
        from src.table_detector import hybrid_detect, spans_to_text, DEFAULT_LAYOUTLM_MODEL
        mp = model_path or DEFAULT_LAYOUTLM_MODEL
        spans = hybrid_detect(
            text           = text,
            pdf_path       = pdf_path,
            model_path     = mp,
            min_confidence = 0.50,
        )
        if spans:
            combined = spans_to_text(spans)
            logger.info(
                f"[TableContent] {len(spans)} blok tabel diekstrak "
                f"({len(combined)} karakter) via LayoutLMv3"
            )
            return combined if combined.strip() else None
    except Exception as e:
        logger.warning(f"[TableContent] LayoutLM deteksi tabel gagal: {e}")
    return None


# ─────────────────────────────────────────────────────────────
# Public API
# ─────────────────────────────────────────────────────────────
def run_ner(
    text: str,
    chunk_size: int = 400,
    pdf_path: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Jalankan NER pada satu teks dokumen.

    Teks panjang dipotong per-kalimat dengan chunk_size token sebagai batas.
    Jika pdf_path diberikan, LayoutLMv3 akan menggunakan bounding boxes PDF
    yang lebih akurat untuk deteksi tabel.

    Args:
        text       : Teks dokumen (hasil PDF extraction).
        chunk_size : Batas token per chunk.
        pdf_path   : Path ke file PDF asli (opsional, untuk LayoutLM).

    Returns:
        Dict dengan key = label NER, value = entitas yang ditemukan.
    """
    _ensure_model()
    nlp    = _get_nlp()
    doc    = nlp(text)

    # Bagi per kalimat, lalu chunk agar tidak melebihi max_len
    all_spans: List[Dict] = []
    chunk_tokens: List[str] = []

    def _flush_chunk():
        nonlocal chunk_tokens
        if not chunk_tokens:
            return
        tags  = _predict_tokens(chunk_tokens)
        spans = extract_spans(chunk_tokens, tags)
        all_spans.extend(spans)
        chunk_tokens = []

    for sent in doc.sents:
        sent_tokens = [tok.text for tok in sent if not tok.is_space]
        if len(chunk_tokens) + len(sent_tokens) > chunk_size:
            _flush_chunk()
        chunk_tokens.extend(sent_tokens)

    _flush_chunk()

    entities = aggregate_entities(all_spans)

    # ── Post-processing: PERIHAL dari text pattern ──────────────
    perihal = _extract_perihal_from_text(text)
    if perihal:
        entities["PERIHAL"] = perihal

    # ── Ekstrak ISI (isi utama dokumen) ─────────────────────────
    if not entities.get("ISI"):
        isi = _extract_isi_from_text(text)
        if isi:
            entities["ISI"] = isi

    # ── Deteksi TABEL dengan LayoutLMv3 ─────────────────────────
    tabel_content = _extract_table_content(text, pdf_path=pdf_path)
    if tabel_content:
        entities["TABEL"] = tabel_content

    return entities


def run_ner_batch(texts: List[str]) -> List[Dict[str, Any]]:
    """Jalankan NER pada beberapa dokumen sekaligus."""
    return [run_ner(t) for t in texts]


### Install Dependencies

In [ ]:
!pip install -r requirement.txt

### Jalankan Training

In [ ]:
!python src/finetune_indobert.py --dataset path_ke_dataset.json --epochs 10 --batch 8